# CO₂ Dehydration for Pipeline Transport

This notebook demonstrates the dehydration of dense-phase CO₂ for pipeline transport
using **TEG (triethylene glycol)** absorption.

## Why Dehydrate CO₂?

For CCS (Carbon Capture & Storage) pipeline transport, CO₂ must be dehydrated to prevent:
- **Hydrate formation** — CO₂ hydrates form readily at pipeline conditions
- **Corrosion** — Wet CO₂ forms carbonic acid (H₂CO₃)
- **Free water drop-out** — Two-phase flow and slugging

Typical water spec: **< 50 ppmv** (stricter than natural gas: 7 lb/MMscf ≈ 112 ppmv)

## Topics Covered
- Water content of CO₂ at various P-T conditions
- CO₂ phase behavior (subcritical, supercritical, dense phase)
- TEG dehydration process simulation
- CO₂-specific dehydration considerations

In [ ]:
# Setup NeqSim
import subprocess, sys
try:
    from neqsim import jneqsim
except ImportError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'neqsim'])
    from neqsim import jneqsim

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

SystemSrkCPAstatoil = jneqsim.thermo.system.SystemSrkCPAstatoil
SystemSrkEos = jneqsim.thermo.system.SystemSrkEos
ThermodynamicOperations = jneqsim.thermodynamicoperations.ThermodynamicOperations
Stream = jneqsim.process.equipment.stream.Stream
ProcessSystem = jneqsim.process.processmodel.ProcessSystem
Separator = jneqsim.process.equipment.separator.Separator
Cooler = jneqsim.process.equipment.heatexchanger.Cooler
Heater = jneqsim.process.equipment.heatexchanger.Heater
Compressor = jneqsim.process.equipment.compressor.Compressor

print('NeqSim loaded successfully')

## 1. Water Content of CO₂ at Pipeline Conditions

Calculate the saturated water content of CO₂ at various pressures and temperatures.
We use the **CPA EOS** which accurately models CO₂-water association.

In [ ]:
# Water content in CO2 at different conditions
pressures = [20, 40, 60, 80, 100, 150, 200]
temperatures = np.arange(0, 65, 5)  # °C

fig, ax = plt.subplots(figsize=(12, 7))
colors = plt.cm.viridis(np.linspace(0.1, 0.9, len(pressures)))

for P, color in zip(pressures, colors):
    water_ppmv = []
    valid_temps = []
    for t in temperatures:
        try:
            fluid = SystemSrkCPAstatoil(273.15 + float(t), float(P))
            fluid.addComponent('CO2', 99.0)
            fluid.addComponent('water', 1.0)
            fluid.setMixingRule('classic_CPA')
            fluid.setMultiPhaseCheck(True)

            ops = ThermodynamicOperations(fluid)
            ops.TPflash()
            fluid.initProperties()

            # Get water in CO2-rich phase
            if fluid.getNumberOfPhases() > 1:
                x_w = float(fluid.getPhase(0).getComponent('water').getx())
                water_ppmv.append(x_w * 1e6)  # ppmv
                valid_temps.append(float(t))
        except Exception:
            pass

    if valid_temps:
        ax.semilogy(valid_temps, water_ppmv, 'o-', color=color,
                    linewidth=2, markersize=5, label=f'{P} bara')

ax.axhline(y=50, color='red', linestyle='--', linewidth=2, alpha=0.7, label='Pipeline spec (50 ppmv)')
ax.set_xlabel('Temperature (°C)', fontsize=12)
ax.set_ylabel('Water Content (ppmv)', fontsize=12)
ax.set_title('Saturated Water Content in CO₂ (CPA EOS)')
ax.legend(title='Pressure', fontsize=10)
ax.grid(True, which='both', alpha=0.3)
ax.set_ylim(1, 1e5)
plt.tight_layout()
plt.savefig('co2_water_content.png', dpi=150, bbox_inches='tight')
plt.show()

## 2. CO₂ Phase Envelope

Generate the phase envelope for a typical CCS CO₂ stream
(with common impurities) to define the operating window for pipeline transport.

In [ ]:
# CO2 phase envelope with impurities
co2_stream = SystemSrkEos(273.15 + 25.0, 80.0)
co2_stream.addComponent('CO2', 96.0)
co2_stream.addComponent('nitrogen', 2.0)
co2_stream.addComponent('methane', 1.0)
co2_stream.addComponent('water', 0.005)  # After dehydration
co2_stream.addComponent('H2S', 0.01)
co2_stream.addComponent('oxygen', 0.5)
co2_stream.setMixingRule('classic')

ops = ThermodynamicOperations(co2_stream)

try:
    ops.calcPTphaseEnvelope(True, 0.0001)

    dewT = list(ops.getOperation().get('dewT'))
    dewP = list(ops.getOperation().get('dewP'))
    bubT = list(ops.getOperation().get('bubT'))
    bubP = list(ops.getOperation().get('bubP'))
    criconT = list(ops.getOperation().get('cricondenthermT'))
    criconP = list(ops.getOperation().get('cricondenthermP'))
    criconbarT = list(ops.getOperation().get('cricondenbarT'))
    criconbarP = list(ops.getOperation().get('cricondenbarP'))

    dewT_C = [float(t) - 273.15 for t in dewT]
    dewP_bar = [float(p) for p in dewP]
    bubT_C = [float(t) - 273.15 for t in bubT]
    bubP_bar = [float(p) for p in bubP]

    fig, ax = plt.subplots(figsize=(10, 7))
    ax.plot(dewT_C, dewP_bar, 'b-', linewidth=2, label='Dew point')
    ax.plot(bubT_C, bubP_bar, 'r-', linewidth=2, label='Bubble point')

    # Mark typical pipeline conditions
    ax.axhspan(80, 200, alpha=0.05, color='green')
    ax.text(20, 120, 'Dense phase\n(pipeline transport)', fontsize=11,
            ha='center', color='green', fontweight='bold')

    ax.set_xlabel('Temperature (°C)', fontsize=12)
    ax.set_ylabel('Pressure (bara)', fontsize=12)
    ax.set_title('Phase Envelope: CCS CO₂ Stream (96% CO₂ + impurities)')
    ax.legend(fontsize=11)
    ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig('co2_phase_envelope.png', dpi=150, bbox_inches='tight')
    plt.show()
except Exception as e:
    print(f'Phase envelope calculation: {e}')
    print('Continuing with other analyses...')

## 3. CO₂ Density in Pipeline Conditions

Understanding CO₂ density is critical for pipeline sizing and hydraulic calculations.

In [ ]:
# CO2 density at pipeline conditions
temps = np.arange(5, 50, 1)
pipeline_pressures = [80, 100, 120, 150, 200]

fig, ax = plt.subplots(figsize=(10, 7))
colors = ['blue', 'green', 'orange', 'red', 'purple']

for P, color in zip(pipeline_pressures, colors):
    densities = []
    valid_t = []
    for t in temps:
        try:
            co2 = SystemSrkEos(273.15 + float(t), float(P))
            co2.addComponent('CO2', 96.0)
            co2.addComponent('nitrogen', 2.0)
            co2.addComponent('methane', 1.0)
            co2.addComponent('oxygen', 1.0)
            co2.setMixingRule('classic')

            ops = ThermodynamicOperations(co2)
            ops.TPflash()
            co2.initProperties()

            rho = float(co2.getDensity('kg/m3'))
            densities.append(rho)
            valid_t.append(float(t))
        except Exception:
            pass

    ax.plot(valid_t, densities, '-', color=color, linewidth=2, label=f'{P} bara')

ax.set_xlabel('Temperature (°C)', fontsize=12)
ax.set_ylabel('Density (kg/m³)', fontsize=12)
ax.set_title('Dense Phase CO₂ Density at Pipeline Conditions')
ax.legend(title='Pressure', fontsize=11)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('co2_pipeline_density.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. TEG Dehydration of CO₂

Simulate TEG dehydration process for wet CO₂, using the same approach as
natural gas dehydration but adapted for dense-phase CO₂.

In [ ]:
import jpype
SimpleTEGAbsorber = jpype.JClass('neqsim.process.equipment.absorber.SimpleTEGAbsorber')

# Wet CO2 feed (saturated with water)
wet_co2 = SystemSrkCPAstatoil(273.15 + 30.0, 110.0)
wet_co2.addComponent('CO2', 98.5)
wet_co2.addComponent('nitrogen', 1.0)
wet_co2.addComponent('water', 0.5)  # Saturated with water
wet_co2.setMixingRule('classic_CPA')

feed = Stream('Wet CO2', wet_co2)
feed.setFlowRate(500000.0, 'kg/hr')  # 500 t/h
feed.setTemperature(30.0, 'C')
feed.setPressure(110.0, 'bara')

# Lean TEG solvent
teg_fluid = SystemSrkCPAstatoil(273.15 + 30.0, 110.0)
teg_fluid.addComponent('TEG', 99.0)  # 99% lean TEG
teg_fluid.addComponent('water', 1.0)
teg_fluid.setMixingRule('classic_CPA')

lean_teg = Stream('Lean TEG', teg_fluid)
lean_teg.setFlowRate(5000.0, 'kg/hr')  # TEG flow rate
lean_teg.setTemperature(30.0, 'C')
lean_teg.setPressure(110.0, 'bara')

dehy_process = ProcessSystem()
dehy_process.add(feed)
dehy_process.add(lean_teg)

# TEG Absorber
absorber = SimpleTEGAbsorber('TEG Absorber')
absorber.addGasInStream(feed)
absorber.addSolventInStream(lean_teg)
absorber.setNumberOfStages(5)
absorber.setStageEfficiency(0.5)
dehy_process.add(absorber)

dehy_process.run()

print('=== TEG Dehydration of Dense-Phase CO₂ ===')
print(f'Wet CO₂ feed:    {feed.getFlowRate("kg/hr"):.0f} kg/hr')
print(f'Feed pressure:   {feed.getPressure():.1f} bara')
print(f'Feed temperature: {feed.getTemperature()-273.15:.1f} °C')

dry_gas = absorber.getGasOutStream().getFluid()
print(f'\nDry CO₂ Composition:')
for i in range(int(dry_gas.getNumberOfComponents())):
    comp = dry_gas.getPhase(0).getComponent(i)
    name = str(comp.getComponentName())
    x = float(comp.getx())
    if name == 'water':
        print(f'  {name:10s}: {x*1e6:.1f} ppmv')
    elif x > 0.001:
        print(f'  {name:10s}: {x*100:.2f} mol%')

print(f'\nDry CO₂ rate: {absorber.getGasOutStream().getFlowRate("kg/hr"):.0f} kg/hr')
print(f'Rich TEG rate: {absorber.getLiquidOutStream().getFlowRate("kg/hr"):.0f} kg/hr')

## 5. TEG Rate Sensitivity Study

Vary the TEG circulation rate to determine the minimum rate needed
to meet the 50 ppmv water specification.

In [ ]:
# TEG rate sensitivity
teg_rates = np.arange(500, 15000, 500)  # kg/hr
water_content = []

for rate in teg_rates:
    try:
        wco2 = SystemSrkCPAstatoil(273.15 + 30.0, 110.0)
        wco2.addComponent('CO2', 98.5)
        wco2.addComponent('nitrogen', 1.0)
        wco2.addComponent('water', 0.5)
        wco2.setMixingRule('classic_CPA')

        f = Stream('WCO2', wco2)
        f.setFlowRate(500000.0, 'kg/hr')
        f.setTemperature(30.0, 'C')
        f.setPressure(110.0, 'bara')

        teg = SystemSrkCPAstatoil(273.15 + 30.0, 110.0)
        teg.addComponent('TEG', 99.0)
        teg.addComponent('water', 1.0)
        teg.setMixingRule('classic_CPA')

        lt = Stream('TEG', teg)
        lt.setFlowRate(float(rate), 'kg/hr')
        lt.setTemperature(30.0, 'C')
        lt.setPressure(110.0, 'bara')

        p = ProcessSystem()
        p.add(f)
        p.add(lt)

        a = SimpleTEGAbsorber('Abs')
        a.addGasInStream(f)
        a.addSolventInStream(lt)
        a.setNumberOfStages(5)
        a.setStageEfficiency(0.5)
        p.add(a)
        p.run()

        x_w = float(a.getGasOutStream().getFluid().getPhase(0).getComponent('water').getx())
        water_content.append(x_w * 1e6)  # ppmv
    except Exception:
        water_content.append(float('nan'))

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(teg_rates/1000, water_content, 'b-o', linewidth=2, markersize=5)
ax.axhline(y=50, color='red', linestyle='--', linewidth=2, label='Pipeline spec (50 ppmv)')
ax.axhline(y=30, color='orange', linestyle='--', linewidth=2, alpha=0.7, label='Strict spec (30 ppmv)')
ax.set_xlabel('TEG Circulation Rate (t/h)', fontsize=12)
ax.set_ylabel('Water Content in Dry CO₂ (ppmv)', fontsize=12)
ax.set_title('CO₂ Dehydration: Water Content vs TEG Rate')
ax.legend(fontsize=11)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('co2_teg_sensitivity.png', dpi=150, bbox_inches='tight')
plt.show()

## Summary

Key findings:

- **Dense-phase CO₂** (>73.8 bara, >31.1°C) is the preferred transport condition
- **Water content** decreases with increasing pressure at constant temperature
- **CPA EOS** is recommended for accurate CO₂-water equilibrium modeling
- **TEG dehydration** can achieve <50 ppmv water in CO₂ with adequate circulation
- Compared to natural gas dehydration, CO₂ requires more care due to:
  - Higher CO₂ solubility in TEG → TEG losses
  - Carbonic acid formation → corrosion concerns
  - Lower water carrying capacity of dense-phase CO₂

### References
- DNV-RP-J202: Design and Operation of CO₂ Pipelines
- ISO 27913: CO₂ Capture, Transport and Geological Storage — Pipeline Transport Systems
- IEAGHG Technical Report: CO₂ Pipeline Infrastructure